In [1]:
import pandas as pd
import scipy.stats
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
from tqdm import tqdm
from geopy.distance import geodesic
from sklearn.preprocessing import TargetEncoder
from concurrent.futures import ThreadPoolExecutor
import itertools
import gc

In [2]:
df = pd.read_csv('data/cleaned_card_transactions_2025.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98037 entries, 0 to 98036
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Recnum             98037 non-null  int64  
 1   Cardnum            98037 non-null  int64  
 2   Date               98037 non-null  object 
 3   Merchnum           98037 non-null  object 
 4   Merch description  98037 non-null  object 
 5   Merch state        98037 non-null  object 
 6   Merch zip          98037 non-null  object 
 7   Transtype          98037 non-null  object 
 8   Amount             98037 non-null  float64
 9   Fraud              98037 non-null  int64  
 10  amount_okay        98037 non-null  bool   
dtypes: bool(1), float64(1), int64(3), object(6)
memory usage: 7.6+ MB


In [4]:
df['Recnum'] = df['Recnum'].astype(str)
df['Cardnum'] = df['Cardnum'].astype(str)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df['amount_okay'] = df['amount_okay'].astype(int)
df = df.drop(columns=['Transtype'])

In [5]:
df['DayOfWeek'] = df['Date'].dt.dayofweek

### Entities Creation

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98037 entries, 0 to 98036
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Recnum             98037 non-null  object        
 1   Cardnum            98037 non-null  object        
 2   Date               98037 non-null  datetime64[ns]
 3   Merchnum           98037 non-null  object        
 4   Merch description  98037 non-null  object        
 5   Merch state        98037 non-null  object        
 6   Merch zip          98037 non-null  object        
 7   Amount             98037 non-null  float64       
 8   Fraud              98037 non-null  int64         
 9   amount_okay        98037 non-null  int32         
 10  DayOfWeek          98037 non-null  int32         
dtypes: datetime64[ns](1), float64(1), int32(2), int64(1), object(6)
memory usage: 7.5+ MB


In [7]:
# Two-way combinations: Card-based
df['card_merch'] = df['Cardnum'] + '_' + df['Merchnum']
df['card_zip'] = df['Cardnum'] + '_' + df['Merch zip']
df['card_state'] = df['Cardnum'] + '_' + df['Merch state']
df['card_merchdesc'] = df['Cardnum'] + '_' + df['Merch description']
df['card_dow'] = df['Cardnum'] + '_' + df['DayOfWeek'].astype(str)

# Two-way combinations: Merchant-based
df['merch_zip'] = df['Merchnum'] + '_' + df['Merch zip']
df['merch_state'] = df['Merchnum'] + '_' + df['Merch state']
df['merchnum_desc'] = df['Merchnum'] + '_' + df['Merch description']
df['merchnum_dow'] = df['Merchnum'] + '_' + df['DayOfWeek'].astype(str)

# Two-way combinations: Location/Description-based
df['state_desc'] = df['Merch state'] + '_' + df['Merch description']
df['merchdesc_dow'] = df['Merch description'] + '_' + df['DayOfWeek'].astype(str)
df['merchdesc_state'] = df['Merch description'] + '_' + df['Merch state']
df['merchdesc_zip'] = df['Merch description'] + '_' + df['Merch zip']

# Three-way combinations: Card + Merchant + Location/Description
df['card_merchnum_desc'] = df['Cardnum'] + '_' + df['Merchnum'] + '_' + df['Merch description']
df['card_merchnum_zip'] = df['Cardnum'] + '_' + df['Merchnum'] + '_' + df['Merch zip']
df['card_merchnum_state'] = df['Cardnum'] + '_' + df['Merchnum'] + '_' + df['Merch state']
df['card_merchdesc_zip'] = df['Cardnum'] + '_' + df['Merch description'] + '_' + df['Merch zip']
df['card_merchdesc_state'] = df['Cardnum'] + '_' + df['Merch description'] + '_' + df['Merch state']

# Three-way combinations: Merchant + Description + Location
df['merchnum_desc_state'] = df['Merchnum'] + '_' + df['Merch description'] + '_' + df['Merch state']
df['merchnum_desc_zip'] = df['Merchnum'] + '_' + df['Merch description'] + '_' + df['Merch zip']

# Zip3-based combinations (these may increase computation time)
df['zip3'] = df['Merch zip'].str[:3]
df['card_zip3'] = df['Cardnum'] + '_' + df['zip3']
df['state_zip'] = df['Merch state'] + '_' + df['Merch zip']
df['merchnum_zip3'] = df['Merchnum'] + '_' + df['zip3']

We are keeping Cardnum and Merchnum as key grouping entities in the following feature engineering. All others are excluded because they are either embedded in the combination entities or is the subject of aggregation (e.g. Amount).

In [8]:
# entity columns for time-based aggregations
entity_cols = [
    'Cardnum', 'Merchnum',
    'card_merch', 'card_zip', 'card_state', 'card_merchdesc', 'card_dow',
    'merch_zip', 'merch_state', 'merchnum_desc', 'merchnum_dow',
    'state_desc', 'merchdesc_dow', 'merchdesc_state', 'merchdesc_zip',
    'card_merchnum_desc', 'card_merchnum_zip', 'card_merchnum_state',
    'card_merchdesc_zip', 'card_merchdesc_state',
    'merchnum_desc_state', 'merchnum_desc_zip',
    'zip3', 'card_zip3', 'state_zip', 'merchnum_zip3'
]

# Filter out entities with low cardinality (<=10 unique values)
entities_to_keep = []
entities_to_remove = []

for entity in entity_cols:
    unique_count = df[entity].nunique()
    status = "REMOVE" if unique_count <= 10 else "KEEP"
    print(f"{entity:30} {unique_count:>8} unique | {status}")
    
    if unique_count <= 10:
        entities_to_remove.append(entity)
    else:
        entities_to_keep.append(entity)

print(f"\nKept {len(entities_to_keep)} entities, removed {len(entities_to_remove)} low-cardinality entities")

Cardnum                            1643 unique | KEEP
Merchnum                          13605 unique | KEEP
card_merch                        40683 unique | KEEP
card_zip                          34236 unique | KEEP
card_state                        16158 unique | KEEP
card_merchdesc                    43669 unique | KEEP
card_dow                           9185 unique | KEEP
merch_zip                         13796 unique | KEEP
merch_state                       13669 unique | KEEP
merchnum_desc                     14588 unique | KEEP
merchnum_dow                      27921 unique | KEEP
state_desc                        13225 unique | KEEP
merchdesc_dow                     27842 unique | KEEP
merchdesc_state                   13225 unique | KEEP
merchdesc_zip                     13467 unique | KEEP
card_merchnum_desc                45861 unique | KEEP
card_merchnum_zip                 40831 unique | KEEP
card_merchnum_state               40712 unique | KEEP
card_merchdesc_zip          

In [9]:
if entities_to_remove:
    df.drop(columns=entities_to_remove, inplace=True)
    print(f"Dropped {len(entities_to_remove)} columns: {entities_to_remove}")

# Update entity list to only include kept entities
entity_cols = entities_to_keep

print(f"dataframe shape: {df.shape}")
print(f"{len(entity_cols)} entities for feature engineering")

dataframe shape: (98037, 35)
26 entities for feature engineering


### Data Partitioning

We use an unconventional naming scheme here: 
- training
- testing
- out-of-time validation set

In [10]:
df.Date.min(), df.Date.max()

(Timestamp('2010-01-01 00:00:00'), Timestamp('2010-12-31 00:00:00'))

We unfortunately only have a full year of data. Ideally, we would want to have at least one year of data in the training + testing partition to account for seasonality. 

However, given the circumstance, we would have to compromise on this and leave some of the months for real out-of-time validation set for true out-of-sample testing that is realistic in a real-time fraud detection algorithm.

In [11]:
df_oot = df[df.Date >= '2010-11-01']
df_train_test = df[df.Date < '2010-11-01']

Ideally, we would employ a time-series cross validation split and engineer features exclusively on the training set to prevent data leakage. However, due to computational constraints, we will first generate features across the entire non-validation partition to facilitate variable selection. Once the final variables are identified, we will rigorously reconstruct the features using only the training partition.

### Target Encoding

Target encoding converts categorical variables into numeric values based on their relationship with the target variable (Fraud). This helps capture the predictive power of high-cardinality categorical features.
- Use smoothing to prevent overfitting on rare categories
- Unseen categories in test data receive the global mean fraud rate

In [12]:
# Columns to target encode
col_TE = ['Merch state', 'Merch zip', 'DayOfWeek']

# Split df into train/test and OOT based on Date
df_oot = df[df['Date'] >= '2010-11-01'].copy()
df_train_test = df[df['Date'] < '2010-11-01'].copy()

print(f"Train/Test partition: {df_train_test.shape[0]} transactions")
print(f"OOT partition: {df_oot.shape[0]} transactions")
print(f"\nColumns to encode: {col_TE}")

Train/Test partition: 85400 transactions
OOT partition: 12637 transactions

Columns to encode: ['Merch state', 'Merch zip', 'DayOfWeek']


In [13]:
for col in col_TE:
    encoder = TargetEncoder(smooth='auto', target_type='binary', random_state=42)
    
    # Fit on train data and transform both train and OOT
    encoder.fit(df_train_test[[col]], df_train_test['Fraud'])
    df_train_test[f'{col}_TE'] = encoder.transform(df_train_test[[col]]).ravel()
    df_oot[f'{col}_TE'] = encoder.transform(df_oot[[col]]).ravel()
    
    print(f"  Train mean: {df_train_test[f'{col}_TE'].mean():.4f}, OOT mean: {df_oot[f'{col}_TE'].mean():.4f}")
    print(f"  Unique values - Train: {df_train_test[col].nunique()}, OOT: {df_oot[col].nunique()}")

print(f"Train/Test shape: {df_train_test.shape}")
print(f"OOT shape: {df_oot.shape}")

  Train mean: 0.0250, OOT mean: 0.0258
  Unique values - Train: 55, OOT: 53
  Train mean: 0.0234, OOT mean: 0.0251
  Unique values - Train: 4379, OOT: 1912
  Train mean: 0.0250, OOT mean: 0.0275
  Unique values - Train: 7, OOT: 7
Train/Test shape: (85400, 38)
OOT shape: (12637, 38)


### Prepare Dataframes for Time-Based Feature Engineering

In [14]:
# df_main: Main dataframe representing "current" transactions
# df_past: Reference dataframe representing "historical/past" transactions for self-joins
# df_features: Accumulator for all engineered features (starts as copy of df_main)

df_main = df_train_test.copy()
df_past = df_train_test.copy()
df_features = df_train_test.copy()

# Add reference columns to df_past for self-join operations
df_past['past_date'] = df_past['Date']
df_past['past_recnum'] = df_past['Recnum']

### Time-Based Feature Engineering

For each entity, we create:
1. **Days since last transaction**: How many days since this entity was last seen
2. **Transaction frequency**: Count of transactions in various time windows (0, 1, 3, 7, 14, 30, 60 days)
3. **Amount statistics**: Average, max, median, and total amounts per time window
4. **Ratio features**: Current amount relative to historical patterns (actual/avg, actual/max, etc.)

In [15]:
# Time windows for rolling aggregations (in days)
TIME_WINDOWS = [0, 1, 3, 7, 14, 30, 60]

# Collect all new features in a dictionary
new_features = {}

for entity in tqdm(entity_cols, desc="Processing entities"):
    
    # Self-join: current transactions with historical transactions
    merged = pd.merge(
        df_main[['Recnum', 'Date', entity]],
        df_past[['past_recnum', 'past_date', entity, 'Amount']],
        on=entity
    )
    
    # Filter to only past transactions (past_recnum < current Recnum)
    past_txns = merged[merged.Recnum > merged.past_recnum]
    
    # DAYS SINCE LAST TRANSACTION
    last_txn = past_txns.groupby('Recnum')[['Date', 'past_date']].last()
    days_since = (last_txn['Date'] - last_txn['past_date']).dt.days
    
    # For those with no prior transaction, fill with days since 2010-01-01
    new_features[f'{entity}_day_since'] = df_main['Recnum'].map(days_since).fillna(
        (df_main['Date'] - pd.to_datetime('2010-01-01')).dt.days
    )
    
    # ROLLING TIME WINDOW FEATURES
    for days in TIME_WINDOWS:
        # Filter transactions within time window
        window_txns = past_txns[
            past_txns['past_date'] >= (past_txns['Date'] - dt.timedelta(days))
        ][['Recnum', entity, 'Amount']]
        
        # Count of transactions in window
        txn_counts = window_txns.groupby('Recnum')[entity].count()
        new_features[f'{entity}_count_{days}'] = df_main['Recnum'].map(txn_counts)
        
        # Amount aggregations
        amount_agg = window_txns.groupby('Recnum')['Amount'].agg(['mean', 'max', 'median', 'sum'])
        new_features[f'{entity}_avg_{days}'] = df_main['Recnum'].map(amount_agg['mean'])
        new_features[f'{entity}_max_{days}'] = df_main['Recnum'].map(amount_agg['max'])
        new_features[f'{entity}_med_{days}'] = df_main['Recnum'].map(amount_agg['median'])
        new_features[f'{entity}_total_{days}'] = df_main['Recnum'].map(amount_agg['sum'])
        
        # Ratio features: current amount relative to historical patterns
        new_features[f'{entity}_actual/avg_{days}'] = df_main['Amount'] / new_features[f'{entity}_avg_{days}']
        new_features[f'{entity}_actual/max_{days}'] = df_main['Amount'] / new_features[f'{entity}_max_{days}']
        new_features[f'{entity}_actual/med_{days}'] = df_main['Amount'] / new_features[f'{entity}_med_{days}']
        new_features[f'{entity}_actual/total_{days}'] = df_main['Amount'] / new_features[f'{entity}_total_{days}']

Processing entities: 100%|██████████| 26/26 [06:07<00:00, 14.15s/it]


In [16]:
print(f"Features in dictionary so far: {len(new_features)}")

Features in dictionary so far: 1664


### Derived Features from Time-Based Aggregations

1. **Velocity ratios**: Compare short-term activity to longer-term patterns
2. **Velocity-to-recency ratios**: Activity rate normalized by days since last transaction  
3. **Amount variability**: Differences between current and past transaction amounts

In [17]:
# VELOCITY RATIOS: Compare short-term vs long-term activity patterns
SHORT_WINDOWS = [0, 1]
LONG_WINDOWS = [7, 14, 30, 60]

for entity in tqdm(entity_cols, desc="Velocity ratios"):
    for short in SHORT_WINDOWS:
        for long in LONG_WINDOWS:
            # Transaction count velocity: short-term count / long-term count / days
            new_features[f'{entity}_count_{short}_by_{long}'] = \
                new_features[f'{entity}_count_{short}'] / new_features[f'{entity}_count_{long}'] / long
            
            # Amount velocity: short-term total / long-term total / days
            new_features[f'{entity}_total_amount_{short}_by_{long}'] = \
                new_features[f'{entity}_total_{short}'] / new_features[f'{entity}_total_{long}'] / long

# VELOCITY-TO-RECENCY RATIOS: Activity rate normalized by days since last transaction
for entity in tqdm(entity_cols, desc="Velocity-to-recency ratios"):
    for short in SHORT_WINDOWS:
        for long in LONG_WINDOWS:
            new_features[f'{entity}_vdratio_{short}by{long}'] = \
                new_features[f'{entity}_count_{short}_by_{long}'] / (new_features[f'{entity}_day_since'] + 1)

Velocity ratios:   0%|          | 0/26 [00:00<?, ?it/s]

Velocity-to-recency ratios: 100%|██████████| 26/26 [00:00<00:00, 268.92it/s]


In [18]:
# AMOUNT VARIABILITY: Track differences between current and historical transaction amounts
VARIABILITY_WINDOWS = [0, 1, 3, 7, 14, 30]

for entity in tqdm(entity_cols, desc="Amount variability"):
    
    merged = pd.merge(
        df_main[['Recnum', 'Date', entity, 'Amount']],
        df_past[['past_recnum', 'past_date', entity, 'Amount']],
        on=entity,
        suffixes=('_current', '_past')
    )
    
    # only past transactions
    past_txns = merged[merged.Recnum > merged.past_recnum]
    
    for days in VARIABILITY_WINDOWS:
        # filter for transactions within time window
        window_txns = past_txns[
            past_txns['past_date'] >= (past_txns['Date'] - dt.timedelta(days))
        ].copy()
        
        window_txns['amount_diff'] = window_txns['Amount_past'] - window_txns['Amount_current']
        
        variability_agg = window_txns.groupby('Recnum')['amount_diff'].agg(['mean', 'max', 'median'])
        new_features[f'{entity}_variability_avg_{days}'] = df_main['Recnum'].map(variability_agg['mean'])
        new_features[f'{entity}_variability_max_{days}'] = df_main['Recnum'].map(variability_agg['max'])
        new_features[f'{entity}_variability_med_{days}'] = df_main['Recnum'].map(variability_agg['median'])

Amount variability: 100%|██████████| 26/26 [03:37<00:00,  8.36s/it]


In [19]:
print(f"Features in dictionary so far: {len(new_features)}")

Features in dictionary so far: 2756


The following cross-entity diversity requires a lot of memory, and we are choosing to save the features we have so far to disk to mitigate memory issues.

In [20]:
print(f"Saving {len(new_features)} features to disk...")

temp_df = pd.DataFrame(new_features, index=df_main.index)
temp_df.to_parquet('data/features_batch_before_crossentity.parquet')
new_features = {}
del temp_df
gc.collect()

Saving 2756 features to disk...


0

### Cross-Entity Diversity Features

Track unique counts of one entity within time windows of another entity.
- Example: "How many unique merchants did this card visit in the past 7 days?"
- Captures diversity and breadth of transaction patterns across entities

In [21]:
# CROSS-ENTITY DIVERSITY: Count unique values of one entity within another entity's time windows
DIVERSITY_WINDOWS = [1, 3, 7, 14, 30, 60]

def process_entity_pair(entity_pair):
    """Process one entity pair for all time windows"""
    entity_i, entity_j = entity_pair
    pair_features = {}
    
    merged = pd.merge(
        df_main[['Recnum', 'Date', entity_i]],
        df_past[['past_recnum', 'past_date', entity_i, entity_j]],
        on=entity_i
    )
    
    # Filter to only past transactions
    past_txns = merged[merged.Recnum > merged.past_recnum]
    
    # Calculate unique counts for each time window
    for days in DIVERSITY_WINDOWS:
        window_txns = past_txns[
            past_txns['past_date'] >= (past_txns['Date'] - dt.timedelta(days))
        ]
        
        # Count unique values of entity_j within entity_i's time window
        unique_counts = window_txns.groupby('Recnum')[entity_j].nunique()
        pair_features[f'{entity_i}_unique_count_for_{entity_j}_{days}'] = unique_counts
    
    # Clean up memory immediately after processing
    del merged, past_txns
    
    return pair_features

# create all entity pairs (excluding same entity)
entity_pairs = [(i, j) for i, j in itertools.product(entity_cols, repeat=2) if i != j]

print(f"Total entity pairs: {len(entity_pairs)}")
print(f"Total features to create: {len(entity_pairs) * len(DIVERSITY_WINDOWS)}")

# batch processing to manage memory
BATCH_SIZE = 20
SAVE_EVERY = 1  # save to disk every N batches
num_batches = (len(entity_pairs) + BATCH_SIZE - 1) // BATCH_SIZE
batch_files = []  # track saved files

for batch_idx in range(num_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(entity_pairs))
    batch = entity_pairs[start_idx:end_idx]
    
    print(f"\nProcessing batch {batch_idx + 1}/{num_batches} ({len(batch)} pairs)...")
    
    with ThreadPoolExecutor(max_workers=2) as executor: 
        results = list(tqdm(
            executor.map(process_entity_pair, batch),
            total=len(batch),
            desc=f"Batch {batch_idx + 1}/{num_batches}"
        ))
    
    # Merge results into new_features dictionary
    batch_features_added = 0
    for result_dict in results:
        for feat_name, feat_values in result_dict.items():
            new_features[feat_name] = df_main['Recnum'].map(feat_values)
            batch_features_added += 1
    
    print(f"Batch {batch_idx + 1} complete. Features added: {batch_features_added}, In memory: {len(new_features)}")
    
    # Save to disk every SAVE_EVERY batches to prevent memory buildup
    if (batch_idx + 1) % SAVE_EVERY == 0:
        filename = f'data/crossentity_batch_{batch_idx + 1}.parquet'
        print(f"Saving {len(new_features)} features to {filename}...")
        temp_df = pd.DataFrame(new_features, index=df_main.index)
        temp_df.to_parquet(filename)
        batch_files.append(filename)
        new_features = {}
        del temp_df
        gc.collect()
    else:
        del results
        gc.collect()

# remaining features - leftover from save_every
if len(new_features) > 0:
    filename = f'data/crossentity_batch_final.parquet'
    print(f"final save: {len(new_features)} features to {filename}...")
    temp_df = pd.DataFrame(new_features, index=df_main.index)
    temp_df.to_parquet(filename)
    batch_files.append(filename)
    new_features = {}
    del temp_df
    gc.collect()

print(f"Saved {len(batch_files)} batch files")

Total entity pairs: 650
Total features to create: 3900

Processing batch 1/33 (20 pairs)...


Batch 1/33: 100%|██████████| 20/20 [01:05<00:00,  3.26s/it]


Batch 1 complete. Features added: 120, In memory: 120
Saving 120 features to data/crossentity_batch_1.parquet...

Processing batch 2/33 (20 pairs)...


Batch 2/33: 100%|██████████| 20/20 [05:41<00:00, 17.06s/it]


Batch 2 complete. Features added: 120, In memory: 120
Saving 120 features to data/crossentity_batch_2.parquet...

Processing batch 3/33 (20 pairs)...


...

Batch 31 complete. Features added: 120, In memory: 120
Saving 120 features to data/crossentity_batch_31.parquet...

Processing batch 32/33 (20 pairs)...


Batch 32/33: 100%|██████████| 20/20 [09:14<00:00, 27.73s/it]


Batch 32 complete. Features added: 120, In memory: 120
Saving 120 features to data/crossentity_batch_32.parquet...

Processing batch 33/33 (10 pairs)...


Batch 33/33: 100%|██████████| 10/10 [03:42<00:00, 22.30s/it]


Batch 33 complete. Features added: 60, In memory: 60
Saving 60 features to data/crossentity_batch_33.parquet...
Saved 33 batch files


We need to reload the saved features from disk for the following squared velocity ratios calculations.

In [24]:
temp_df = pd.read_parquet('data/features_batch_before_crossentity.parquet')

new_features = temp_df.to_dict('series')
del temp_df
gc.collect()

177

In [25]:
# SQUARED VELOCITY RATIOS: Velocity normalized by squared time windows
# This gives more weight to recent activity relative to longer-term patterns

for entity in tqdm(entity_cols, desc="Squared velocity ratios"):
    for short in SHORT_WINDOWS:
        for long in LONG_WINDOWS:
            new_features[f'{entity}_count_{short}_by_{long}_sq'] = \
                new_features[f'{entity}_count_{short}'] / new_features[f'{entity}_count_{long}'] / (long ** 2)

Squared velocity ratios: 100%|██████████| 26/26 [00:00<00:00, 112.97it/s]


### Additional Business Logic Features

Domain-specific fraud indicators based on transaction patterns and geography.

In [26]:
# GEOGRAPHIC DISTANCE FEATURES
# Calculate distance between consecutive transactions for each card
    
# Load ZIP code database with coordinates
zip_df = pd.read_csv("data/zip_code_database.csv")
zip_df['zip'] = zip_df['zip'].astype(str)

# Create ZIP to coordinates mapping
zip_to_coords = zip_df.set_index('zip')[['latitude', 'longitude']].to_dict('index')

def calculate_distance(zip1, zip2):
    """Calculate distance in miles between two ZIP codes"""
    coords_1 = zip_to_coords.get(str(zip1))
    coords_2 = zip_to_coords.get(str(zip2))
    if coords_1 and coords_2:
        return geodesic(
            (coords_1['latitude'], coords_1['longitude']), 
            (coords_2['latitude'], coords_2['longitude'])
        ).miles
    return None

# Get previous transaction ZIP for each card
df_temp = df_main.copy()
df_temp['prev_merch_zip'] = df_temp.groupby('Cardnum')['Merch zip'].shift(1)

# Calculate distance to last transaction
new_features['distance_to_last_transaction'] = df_temp.apply(
    lambda row: calculate_distance(row['prev_merch_zip'], row['Merch zip']), 
    axis=1
)

# Flag suspicious distances (>1000 miles)
new_features['suspicious_distance_flag'] = (
    new_features['distance_to_last_transaction'] > 1000
).astype(int)

In [27]:
# AMOUNT CATEGORY BINS
# Discretize amounts into quintiles for categorical analysis

new_features['amount_category'] = pd.qcut(
    df_main['Amount'], 
    q=5, 
    labels=[1, 2, 3, 4, 5]
).astype(int)

print(new_features['amount_category'].value_counts().sort_index())

Amount
1    17080
2    17080
3    17080
4    17105
5    17055
Name: count, dtype: int64


In [28]:
# FOREIGN ZIP CODE FLAG
# Identify transactions at merchants with non-US ZIP codes

us_zips = set(zip_df['zip'].astype(str).values)

new_features['foreign_zip_flag'] = df_main['Merch zip'].apply(
    lambda x: 0 if str(x) in us_zips else 1
)

print(f"Foreign ZIP transactions: {new_features['foreign_zip_flag'].sum()}")

Foreign ZIP transactions: 8923


In [29]:
# WEEKEND TRANSACTION FLAG
# Binary indicator for transactions on Saturday or Sunday

new_features['is_weekend'] = (df_main['DayOfWeek'] >= 5).astype(int)

print(f"Weekend transactions: {new_features['is_weekend'].sum()} ({new_features['is_weekend'].mean()*100:.1f}%)")

Weekend transactions: 25075 (29.4%)


In [30]:
# ROUNDED AMOUNT FLAG
# Flag whole number amounts (potentially suspicious pattern)

new_features['amount_is_rounded'] = (df_main['Amount'] % 1 == 0).astype(int)

print(f"Rounded amounts: {new_features['amount_is_rounded'].sum()} ({new_features['amount_is_rounded'].mean()*100:.1f}%)")

Rounded amounts: 22300 (26.1%)


In [31]:
# HIGH AMOUNT FOR MERCHANT FLAG
# Flag transactions exceeding 2x the merchant's average amount

merchant_avg = df_main.groupby('Merchnum')['Amount'].mean()
df_main['merchant_avg_amount'] = df_main['Merchnum'].map(merchant_avg)

new_features['high_amount_for_merchant'] = (
    df_main['Amount'] > (2 * df_main['merchant_avg_amount'])
).astype(int)

print(f"High amount transactions: {new_features['high_amount_for_merchant'].sum()} ({new_features['high_amount_for_merchant'].mean()*100:.1f}%)")

High amount transactions: 8348 (9.8%)


For each card, what percentage of its transactions go to its single most-visited merchant?

In [32]:
# MERCHANT DOMINANCE SCORE
# Fraction of card's transactions at its most frequent merchant

card_merchant_counts = df_main.groupby(['Cardnum', 'Merchnum']).size().reset_index(name='txn_count')
most_freq_merchant = card_merchant_counts.loc[
    card_merchant_counts.groupby('Cardnum')['txn_count'].idxmax()
]

total_txns_per_card = df_main.groupby('Cardnum').size().reset_index(name='total_txns')

dominance = pd.merge(most_freq_merchant, total_txns_per_card, on='Cardnum')
dominance['merchant_dominance_score'] = dominance['txn_count'] / dominance['total_txns']

new_features['merchant_dominance_score'] = df_main['Cardnum'].map(
    dominance.set_index('Cardnum')['merchant_dominance_score']
)

print(f"Mean dominance score: {new_features['merchant_dominance_score'].mean():.3f}")

Mean dominance score: 0.256


In [33]:
# STATE INCONSISTENCY FLAG
# Flag if card used in different state within 1 day (impossible travel)

df_sorted = df_main.sort_values(['Cardnum', 'Date'])
df_sorted['prev_state'] = df_sorted.groupby('Cardnum')['Merch state'].shift(1)
df_sorted['prev_date'] = df_sorted.groupby('Cardnum')['Date'].shift(1)
df_sorted['days_since_prev'] = (df_sorted['Date'] - df_sorted['prev_date']).dt.days

new_features['state_inconsistency_flag'] = (
    (df_sorted['Merch state'] != df_sorted['prev_state']) & 
    (df_sorted['days_since_prev'] <= 1)
).fillna(False).astype(int)

print(f"State inconsistencies: {new_features['state_inconsistency_flag'].sum()} ({new_features['state_inconsistency_flag'].mean()*100:.1f}%)")

State inconsistencies: 27266 (31.9%)


### Final Feature Concatanation

In [34]:
features_list = []

# before cross-entity
features_list.append(pd.read_parquet('data/features_batch_before_crossentity.parquet'))

# cross-entity batch files
if 'batch_files' in globals() and batch_files:
    for filepath in batch_files:
        features_list.append(pd.read_parquet(filepath))

# business logic features from new_features dictionary
if len(new_features) > 0:
    features_list.append(pd.DataFrame(new_features, index=df_main.index))

all_features_df = pd.concat(features_list, axis=1)
del features_list
gc.collect()

base_cols = ['Recnum', 'Date', 'Cardnum', 'Merchnum', 'Amount', 'Fraud', 
             'Merch description', 'Merch state', 'Merch zip', 'DayOfWeek']
df_features = pd.concat([df_main[base_cols], all_features_df], axis=1)
del all_features_df
gc.collect()

print(f"shape: {df_features.shape}")

shape: (85400, 9639)


Remove duplicated columns.

In [40]:
df_features = df_features.loc[:, ~df_features.columns.duplicated()]
print(f"shape: {df_features.shape}")

shape: (85400, 6883)


Remove first two weeks of data.

In [42]:
df_features = df_features[df_features['Date'] >= df_features['Date'].min() + pd.Timedelta(days=14)].reset_index(drop=True)

print(f"new date range: {df_features['Date'].min()} to {df_features['Date'].max()}")
print(f"new shape: {df_features.shape}")

new date range: 2010-01-15 00:00:00 to 2010-10-31 00:00:00
new shape: (81892, 6883)


The nulls are the natural result of calculating historical features when no history exists yet. We fill them with 0s.

In [44]:
df_features.fillna(0, inplace=True)

The following columns should not be used as features.

In [47]:
# df_features.drop(columns=[['Recnum', 'Date', 'Cardnum', 'Merchnum', 'Amount', 'Fraud', 
#             'Merch description', 'Merch state', 'Merch zip', 'DayOfWeek']], inplace=True)

### Final Ouput

In [48]:
df.to_parquet("data/cleaned_card_transactions_with_entities.parquet")
df_features.to_parquet("data/candidate_variables.parquet")